base code

In [1]:
# from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# model_name = "google/flan-t5-small"

# # Load tokenizer + model
# tokenizer = AutoTokenizer.from_pretrained(model_name)
# model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# # Prompt (important!)
# prompt = """Classify sentiment as Positive or Negative.
# Text: I love this product.
# Answer:"""

# # Tokenize
# inputs = tokenizer(prompt, return_tensors="pt")

# # Generate output
# outputs = model.generate(
#     **inputs,
#     max_new_tokens=5,
#     do_sample=False
# )

# # Decode result
# result = tokenizer.decode(outputs[0], skip_special_tokens=True)

# print(result)

New code

In [2]:
from sklearn.metrics import f1_score
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

/Users/negin/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/negin/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [17]:
# Load the dataset
valid_df = pd.read_csv('data/6/valid.csv')
valid_df.drop(columns=['data_id'], inplace=True)

In [32]:
# preprocess text (clean)
def clean_text(df, text_col="text", target_col="sentiment"):

    df = df.dropna(subset=[text_col])
    df[text_col] = df[text_col].str.strip()
    df[text_col] = df[text_col].str[:2048]

    df[target_col] = df[target_col].str.lower()
    return df

df = clean_text(valid_df)

In [33]:
def batch_predict(texts):
    prompts = [
        f"Classify sentiment as Positive or Negative.\nText: {t}\nAnswer:"
        for t in texts
    ]

    inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False
        )

    return tokenizer.batch_decode(outputs, skip_special_tokens=True)

    
# Load model
model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Set model to eval mode (faster)
model.eval()

# Apply in batches
batch_size = 32
preds = []

for i in range(0, len(df), batch_size):
    batch_texts = df["text"].iloc[i:i+batch_size].tolist()
    preds.extend(batch_predict(batch_texts))

df["predicted"] = [p.strip() for p in preds]

In [34]:
df['predicted'] = df['predicted'].str.lower()
df['sentiment'] = df['sentiment'].str.lower()
df['correct'] = df['sentiment'] == df['predicted']
# Calculate accuracy
accuracy = df['correct'].mean()
print(f"Accuracy: {accuracy:.2%}")

# Calculate F1-score
f1 = f1_score(df['sentiment'], df['predicted'], average='weighted')
print(f"F1-Score: {f1}")

Accuracy: 87.14%
F1-Score: 0.8792653061224489
